In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import h5netcdf
import matplotlib.colors as mcolors
import pandas as pd
from matplotlib.font_manager import FontProperties
from matplotlib.patches import Wedge, Patch
import matplotlib.cm as cm

In [ ]:
model_names = ['CESM2-WACCM', 'IPSL-CM6A-LR', 'MPI-ESM1-2-LR', 'MPI-ESM1-2-HR', 'UKESM1-0-LL', 'CNRM-ESM2-1']

In [ ]:
#future KTC datasets
future_paths = [ r"D:\2085-99 KTC Data\CESM2-WACCM_complete_KTC_SSP2-4.5_dataset_2085-2099.nc",
               r"D:\2085-99 KTC Data\CNRM-ESM2-1_complete_KTC_SSP2-4.5_dataset_2085-2099.nc",
               r"D:\2085-99 KTC Data\IPSL-CM6A-LR_complete_KTC_SSP2-4.5_dataset_2085-2099.nc",
               r"D:\2085-99 KTC Data\MPI-ESM1-2-LR_complete_KTC_SSP2-4.5_dataset_2085-2099.nc",
               r"D:\2085-99 KTC Data\MPI-ESM1-2-HR_complete_KTC_SSP2-4.5_dataset_2085-2099.nc",
               r"D:\2085-99 KTC Data\UKESM1-0-LL_complete_KTC_SSP2-4.5_dataset_2085-2099.nc" ]

#Historical KTC datasets
historical_paths = [ r"D:\Historical KTC Datasets\CESM2-WACCM_KTC_Historical_dataset",
                   r"D:\Historical KTC Datasets\CNRM-ESM2-1_KTC_Historical_dataset",
                   r"D:\Historical KTC Datasets\IPSL-CM6A-LR_KTC_Historical_dataset",
                   r"D:\Historical KTC Datasets\MPI-ESM1-2-LR_KTC_Historical_dataset",
                   r"D:\Historical KTC Datasets\MPI-ESM1-2-HR_KTC_Historical_dataset",
                   r"D:\Historical KTC Datasets\UKESM1-0-LL_KTC_Historical_dataset" ]

#area datasets
area_paths = [ r"D:\CESM2-WACCM Data\areacella_fx_CESM2-WACCM_G6sulfur_r1i1p1f2_gn.nc",
             r"D:\CNRM Data\areacella_fx_CNRM-ESM2-1_historical_r11i1p1f2_gr.nc",
             r"D:\IPSL Data\areacella_fx_IPSL-CM6A-LR_G6sulfur_r1i1p1f1_gr.nc",
             r"D:\MPI LR Data\areacella_fx_MPI-ESM1-2-LR_ssp245_r11i1p1f1_gn.nc",
             r"D:\MPI HR Data\areacella_fx_MPI-ESM1-2-HR_G6sulfur_r1i1p1f1_gn.nc",
             r"D:\UKESM Data\areacella_fx_UKESM1-0-LL_piControl_r1i1p1f2_gn.nc" ]

#land frac datasets
land_paths = [ r"D:\CESM2-WACCM Data\sftlf_fx_CESM2-WACCM_G6sulfur_r1i1p1f2_gn.nc",
             r"D:\CNRM Data\sftlf_fx_CNRM-ESM2-1_historical_r11i1p1f2_gr.nc",
             r"D:\IPSL Data\sftlf_fx_IPSL-CM6A-LR_G6sulfur_r1i1p1f1_gr.nc",
             r"D:\MPI LR Data\sftlf_fx_MPI-ESM1-2-LR_ssp245_r11i1p1f1_gn.nc",
             r"D:\MPI HR Data\sftlf_fx_MPI-ESM1-2-HR_G6sulfur_r1i1p1f1_gn.nc",
             r"D:\UKESM Data\sftlf_fx_UKESM1-0-LL_piControl_r1i1p1f2_gn.nc" ]

In [ ]:
datasets = []
for i in range(6):
    future_data = xr.open_dataset(future_paths[i])
    historical_data = xr.open_dataset(historical_paths[i]) #force it to open as netcdf file bc it doesn't have .nc backend
    area_data = xr.open_dataset(area_paths[i])
    land_data = xr.open_dataset(land_paths[i])
    
    datasets.append({
        "model": model_names[i],
        "future": future_data,
        "historical": historical_data,
        "area": area_data,
        "landfrac": land_data
    })

In [ ]:
#specify all landfrac thresholds to apply
land_thresh = {
    'CESM2-WACCM': 50,
    'CNRM-ESM2-1': 45,
    'IPSL-CM6A-LR': 30,
    'MPI-ESM1-2-LR': 35,
    'MPI-ESM1-2-HR': 50,
    'UKESM1-0-LL': 40
}

In [ ]:
#specify transitions --> unhashtag to use

#MAJOR WARM 
#target_transitions = [ "A->xA", "C->xA", "C->A", "D->C", "F->C", "E->D", "F->D","F->E" ]

#MAJOR DRY
#target_transitions = [ "A->B", "C->B", "D->B", "E->B", "B->C" ]

#MAJOR WET
#target_transitions = [ "B->xA", "B->A" ]

#there are no major cold transitions

#MINOR WARM
#target_transitions = ['BWk->BWh', 'BSk->BSh', 'DC->DO', 'Fi->Ft']

#MINOR COLD
#target_transitions = ['BSh->BSk','DO->DC', 'Ft->Fi']

#MINOR DRY
#target_transitions = ['BSh->BWh', 'BSh->BWk', 'BSk->BWh','BSk->BWk', 'Ar->Aw','Cf->Cm']

#MINOR WET
#target_transitions = ['BWh->BSh', 'BWh->BSk', 'BWk->BSh','BWk->BSk','Aw->Ar', 'Cm->Cf']

percent_results = []

for i, data in enumerate(datasets): #enumerate loops thru models
    #loop thru all data
    model = data['model']
    hist_ds = data['historical']
    future_ds = data['future']
    area_ds = data['area']
    land_ds = data['landfrac']
    #get global area (area_land)
    area_grid = area_ds['areacella']
    land_grid = land_ds['sftlf']
    threshold = land_thresh.get(model)
    land_mask = land_grid > threshold
    area_land = area_grid.where(land_mask)

    #align all data perfectly
    historical = hist_ds.interp_like(area_land)
    future = future_ds.interp_like(area_land)

    #create xA - for major climate transitions only
    #also if KTC data doesn't already have xA variable
   # if "xAr" in future and "xAw" in future:
        #future["xA"] = (future["xAr"] + future["xAw"]).astype(bool)

    #create zone masks for historical and future datasets
    #maps each zone label to the corresponding hist/fut boolean mask from hist/fut ds
    #zone_labels = ["xA", "A", "B", "C", "D", "E", "F"] #MAJOR
    #zone_labels = ['BWh', 'BWk', 'BSh', 'BSk', 'Ar', 'Aw', 'Cf', 'Cm', 'DC', 'DO', 'Ft', 'Fi'] #MINOR
    zones_h = {z: historical[z] for z in zone_labels if z in historical}
    zones_f = {z: future[z] for z in zone_labels if z in future}
    #z takes on the zones one at a time (A, B, etc)
    #For each label z in zone_labels,
    #if z exists in the historical dataset,
    #then include it in the output dictionary,
    #where the key is z and the value is historical[z]

    #collapse to str labels --> out is completed output
    def collapse(zone_dict, order): #collapse into 2D output
    #--> where zone_dict is an array of bool zones (ex: 'A': 2D array where A is true)
        #order just specifies the priority in which zones are assigned to final output (same as zone_labels)
        first = list(zone_dict.keys())[0] #gets any zone label, allows to grab shape of 2D grid
        shape = zone_dict[first].shape #gets shape of any single zone mask (first one it matches to)
        out = np.full(shape, '', dtype=object) #creates empty str array 
        #each grid cell will hold the zone label it belongs to 
        for label in order:
            if label in zone_dict:
                mask = zone_dict[label].values.astype(bool) #convert to bool
                out[mask] = label
        return out

    #zone_labels = order (order is defined here, zone_dict is zones_h/f, also defined)
    z20_str = collapse(zones_h, zone_labels)
    z90_str = collapse(zones_f, zone_labels)
    #add arrow, make str label nice, concatenate hist + fut labels
    transitions = np.char.add(np.char.add(z20_str, "->"), z90_str)

    #total land area (sum)
    total_land = area_land.sum().item()

    #area of transition zones
    transition_area = 0.0
    for t in target_transitions:
        mask = transitions == t
        area_sum = area_land.where(mask).sum().item()
        transition_area += area_sum

        #compute percents
    percent = (transition_area / total_land) * 100
    percent_results.append((model, percent))

    print(f"{model}: {percent:.2f}%")

#compute avg over all models
avg_percent = np.mean([p for _, p in percent_results])
print(f"\nAverage across all models: {avg_percent:.2f}%")

#store these numbers for later to input into the piecharts (where it says [fill in pcts here])

In [ ]:
percents_sets = [
    [fill in pcts here], #ssp2-4.5
    [fill in pcts here], #ssp5-8.5
    [fill in pcts here] #g6sulfur
] # make sure pcts are in correct order
names = ['Major Hot', 'Minor Hot', 'Major Dry', 'Minor Dry', 'Major Wet', 'Minor Wet']
colors = ['mediumvioletred', 'palevioletred', 'goldenrod', 'gold', 'darkorchid', 'violet']

def absolute_value(pct, allvals):
    total = sum(allvals)
    return f"{pct*total/100:.2f}%"

#scale each chart -- use total percentage as proxy
totals = [sum(p) for p in percents_sets]
max_total = max(totals)
radii = [0.5 + 0.5*(t/max_total) for t in totals]   # radii from 0.5 to 1.0

fig, axes = plt.subplots(1, 3, figsize=(12, 4), dpi=600)
fig.subplots_adjust(bottom=0.25)

for ax, percents, r in zip(axes, percents_sets, radii):
    wedges, texts, autotexts = ax.pie(
        percents,
        colors=colors,
        startangle=90,
        autopct=lambda pct: absolute_value(pct, percents),
        radius=r, 
         )
    x, y = autotexts[4].get_position() #adjust text for major wet
    autotexts[4].set_position((x, y - 0.06))
    x, y = autotexts[3].get_position() #adjust text for minor dry
    autotexts[3].set_position((x, y - 0.05))

    for autotext in autotexts:
        autotext.set_fontsize(8)
        autotext.set_fontfamily('Times New Roman')
        autotext.set_color('white')
        autotext.set_fontweight('bold')
    ax.set_aspect('equal')  #keep circles round


titles = ['SSP2-4.5', 'SSP5-8.5', 'G6sulfur']
for ax, title in zip(axes, titles):
    ax.set_title(title, fontsize=14, fontfamily='Times New Roman', y=0.95)

plt.tight_layout()
plt.savefig('insert image path here', dpi=600, bbox_inches='tight')
plt.show()